In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pingouin as pg

sns.set_theme(style="whitegrid", context="talk")

In [ ]:
PROJECT_ROOT = Path("/home/duarte/Desktop/Tese/Mapping_Tese/mapping_tese")

SVC_CSV = PROJECT_ROOT / "notebooks/SVC/results/svc_multi_subject_summary.csv"
ANN_CSV = PROJECT_ROOT / "notebooks/ANN/results/4_Classes_ANN_MultiSubject/all_subjects_results.csv"
OUT_DIR = PROJECT_ROOT / "images/SVC_vs_ANN"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("SVC CSV:", SVC_CSV)
print("ANN CSV:", ANN_CSV)
print("Output folder:", OUT_DIR)
print("SVC exists:", SVC_CSV.exists())
print("ANN exists:", ANN_CSV.exists())

In [ ]:
svc_df = pd.read_csv(SVC_CSV)[["subject", "mean_balanced_accuracy"]].rename(
    columns={"mean_balanced_accuracy": "svc_bal_acc"}
)

ann_df = pd.read_csv(ANN_CSV)[["subject", "mean_balanced_accuracy"]].rename(
    columns={"mean_balanced_accuracy": "ann_bal_acc"}
)

merged = svc_df.merge(ann_df, on="subject", how="inner", validate="one_to_one")

merged["svc_bal_acc_pct"] = merged["svc_bal_acc"] * 100.0
merged["ann_bal_acc_pct"] = merged["ann_bal_acc"] * 100.0
merged["delta_ann_minus_svc"] = merged["ann_bal_acc"] - merged["svc_bal_acc"]
merged["delta_ann_minus_svc_pct"] = merged["delta_ann_minus_svc"] * 100.0

eps = 1e-12
merged["winner"] = np.select(
    [
        merged["delta_ann_minus_svc"] > eps,
        merged["delta_ann_minus_svc"] < -eps,
    ],
    ["ANN", "SVC"],
    default="Tie",
)

print("Subjects in merged table:", len(merged))
display(merged.sort_values("subject").reset_index(drop=True))

In [ ]:
SVC_RESULTS_DIR = PROJECT_ROOT / "notebooks/SVC/results"
ANN_RESULTS_DIR = PROJECT_ROOT / "notebooks/ANN/results/4_Classes_ANN_MultiSubject"

def load_fold_balanced_accuracy(base_dir, model_name):
    rows = []

    for csv_path in sorted(base_dir.glob("sub-*/fold_balanced_accuracy.csv")):
        df = pd.read_csv(csv_path)
        needed = {"subject", "fold", "balanced_accuracy"}
        missing = needed - set(df.columns)
        if missing:
            raise ValueError(f"{csv_path} is missing columns: {sorted(missing)}")
        df = df[["subject", "fold", "balanced_accuracy"]].copy()
        df["model"] = model_name
        rows.append(df)

    if not rows:
        return pd.DataFrame(columns=["subject", "fold", "balanced_accuracy", "model"])

    out = pd.concat(rows, ignore_index=True)
    out["fold"] = out["fold"].astype(int)
    out["balanced_accuracy"] = out["balanced_accuracy"].astype(float)
    return out

svc_fold_df = load_fold_balanced_accuracy(SVC_RESULTS_DIR, "SVC")
ann_fold_df = load_fold_balanced_accuracy(ANN_RESULTS_DIR, "ANN")

if svc_fold_df.empty or ann_fold_df.empty:
    raise FileNotFoundError(
        "No fold_balanced_accuracy.csv files found. "
        "Rerun the SVC/ANN training notebook cells that save per-fold balanced accuracy."
    )

shared_subjects = sorted(set(svc_fold_df["subject"]) & set(ann_fold_df["subject"]))
if not shared_subjects:
    raise ValueError("No overlapping subjects with fold-level files between SVC and ANN.")

fold_plot_df = pd.concat(
    [
        svc_fold_df[svc_fold_df["subject"].isin(shared_subjects)],
        ann_fold_df[ann_fold_df["subject"].isin(shared_subjects)],
    ],
    ignore_index=True,
)

fold_plot_df["balanced_accuracy_pct"] = fold_plot_df["balanced_accuracy"] * 100.0
fold_plot_df["model"] = pd.Categorical(
    fold_plot_df["model"], categories=["SVC", "ANN"], ordered=True
)

fold_plot_df = fold_plot_df.sort_values(["subject", "model", "fold"]).reset_index(drop=True)
fold_plot_df.to_csv(OUT_DIR / "per_fold_balanced_accuracy_long.csv", index=False)

check_counts = (
    fold_plot_df.groupby(["subject", "model"])["fold"]
    .nunique()
    .reset_index(name="n_folds")
)
bad = check_counts[check_counts["n_folds"] != 4]
if not bad.empty:
    print("Warning: some subject/model pairs do not have exactly 4 folds:")
    display(bad)

display(fold_plot_df.head(12))
print("Shared subjects:", shared_subjects)
print("Rows in fold dataframe:", len(fold_plot_df))
print("Expected rows (shared_subjects * 4 * 2):", len(shared_subjects) * 4 * 2)

In [ ]:
if fold_plot_df.empty:
    raise ValueError("fold_plot_df is empty. Run the fold-loading cell first.")

palette = {"SVC": "#4C78A8", "ANN": "#F58518"}

fold_plot_df_plot = fold_plot_df.copy()
fold_plot_df_plot["_group"] = "All subjects"

fig, ax = plt.subplots(figsize=(5, 6), dpi=150)

sns.violinplot(
    data=fold_plot_df_plot,
    x="_group",
    y="balanced_accuracy_pct",
    hue="model",
    split=True,
    palette=palette,
    inner="quart",
    cut=0,
    linewidth=1.2,
    ax=ax,
)

ax.axhline(25, color="red", linestyle="--", linewidth=1.2, label="Chance (25%)")
ax.set_xlabel("")
ax.set_ylabel("Balanced Accuracy (%)")
ax.set_title("Fold Balanced Accuracy: SVC vs ANN\n(11 subjects × 4 folds each)")
ax.legend(title="Model", loc="upper right", frameon=True)

plt.tight_layout()
plt.savefig(OUT_DIR / "violin_fold_balanced_accuracy_svc_vs_ann.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
per_subject_table = merged[
    ["subject", "svc_bal_acc_pct", "ann_bal_acc_pct", "delta_ann_minus_svc_pct", "winner"]
].copy()

per_subject_table = per_subject_table.sort_values("delta_ann_minus_svc_pct", ascending=False).reset_index(drop=True)
per_subject_table_rounded = per_subject_table.copy()
per_subject_table_rounded[["svc_bal_acc_pct", "ann_bal_acc_pct", "delta_ann_minus_svc_pct"]] = \
    per_subject_table_rounded[["svc_bal_acc_pct", "ann_bal_acc_pct", "delta_ann_minus_svc_pct"]].round(2)

per_subject_table_rounded.to_csv(OUT_DIR / "per_subject_balanced_accuracy_table_rounded.csv", index=False)
display(per_subject_table_rounded)

print("ANN wins:", int((per_subject_table["winner"] == "ANN").sum()))
print("SVC wins:", int((per_subject_table["winner"] == "SVC").sum()))
print("Ties:", int((per_subject_table["winner"] == "Tie").sum()))
print("Mean delta (ANN - SVC), percentage points:", round(per_subject_table["delta_ann_minus_svc_pct"].mean(), 2))

In [ ]:
x = merged["svc_bal_acc"].to_numpy(dtype=float)
y = merged["ann_bal_acc"].to_numpy(dtype=float)
d = y - x

normality_df = pg.normality(d)
ttest_df = pg.ttest(y, x, paired=True)
wilcoxon_df = pg.wilcoxon(y, x, alternative="two-sided")

normality_df.to_csv(OUT_DIR / "stats_normality_diff.csv", index=False)
ttest_df.to_csv(OUT_DIR / "stats_paired_ttest.csv", index=False)
wilcoxon_df.to_csv(OUT_DIR / "stats_wilcoxon.csv", index=False)

def pick_value(df, candidates, default=np.nan):
    if df.empty:
        return float(default)
    for c in candidates:
        if c in df.columns:
            try:
                return float(df[c].iloc[0])
            except Exception:
                return float(default)
    return float(default)

paired_t_p = pick_value(ttest_df, ["p-val", "p_val", "pvalue", "p"])
paired_t_d = pick_value(ttest_df, ["cohen-d", "cohen_d", "cohend", "d"])
wilcoxon_p = pick_value(wilcoxon_df, ["p-val", "p_val", "pvalue", "p"])

summary_df = pd.DataFrame([{
    "n_subjects": len(merged),
    "svc_mean_bal_acc": float(np.mean(x)),
    "svc_std_bal_acc": float(np.std(x, ddof=1)),
    "ann_mean_bal_acc": float(np.mean(y)),
    "ann_std_bal_acc": float(np.std(y, ddof=1)),
    "mean_diff_ann_minus_svc": float(np.mean(d)),
    "mean_diff_percent_points": float(np.mean(d) * 100.0),
    "paired_t_p": paired_t_p,
    "paired_t_cohen_d": paired_t_d,
    "wilcoxon_p": wilcoxon_p,
}])

summary_df.to_csv(OUT_DIR / "balanced_accuracy_stats_summary.csv", index=False)

print("Normality test of paired differences:")
display(normality_df)
print("Paired t-test:")
display(ttest_df)
print("Wilcoxon signed-rank:")
display(wilcoxon_df)
print("Compact summary:")
display(summary_df)

In [ ]:
plot_df = merged.sort_values("delta_ann_minus_svc_pct", ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 0.6 * len(plot_df) + 2), dpi=150)
y_pos = np.arange(len(plot_df))

for i, row in plot_df.iterrows():
    ax.plot(
        [row["svc_bal_acc_pct"], row["ann_bal_acc_pct"]],
        [i, i],
        color="gray",
        alpha=0.7,
        linewidth=2
    )

ax.scatter(plot_df["svc_bal_acc_pct"], y_pos, s=70, color="#4C78A8", label="SVC", zorder=3)
ax.scatter(plot_df["ann_bal_acc_pct"], y_pos, s=70, color="#F58518", label="ANN", zorder=3)

ax.axvline(25, color="red", linestyle="--", linewidth=1.2, label="Chance (25%)")

ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df["subject"])
ax.set_xlabel("Balanced Accuracy (%)")
ax.set_title("Per-subject Balanced Accuracy: SVC vs ANN")
ax.legend(loc="lower right", frameon=True)

plt.tight_layout()
plt.savefig(OUT_DIR / "per_subject_dumbbell_svc_vs_ann.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
delta_df = merged.sort_values("delta_ann_minus_svc_pct", ascending=False).reset_index(drop=True)
colors = np.where(delta_df["delta_ann_minus_svc_pct"] >= 0, "#2E8B57", "#B22222")

fig, ax = plt.subplots(figsize=(12, 6), dpi=150)
bars = ax.bar(delta_df["subject"], delta_df["delta_ann_minus_svc_pct"], color=colors, alpha=0.9)

ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Delta (ANN - SVC), percentage points")
ax.set_title("Per-subject Change in Balanced Accuracy")
ax.tick_params(axis="x", rotation=45)

for b, val in zip(bars, delta_df["delta_ann_minus_svc_pct"]):
    ax.text(
        b.get_x() + b.get_width() / 2,
        val + (0.4 if val >= 0 else -0.6),
        f"{val:.1f}",
        ha="center",
        va="bottom" if val >= 0 else "top",
        fontsize=9
    )

plt.tight_layout()
plt.savefig(OUT_DIR / "per_subject_delta_ann_minus_svc.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
lines = []
lines.append("Per-subject SVC vs ANN (Balanced Accuracy)")
lines.append("")

for _, r in per_subject_table_rounded.iterrows():
    lines.append(
        f"{r['subject']}: "
        f"SVC={r['svc_bal_acc_pct']:.2f}%, "
        f"ANN={r['ann_bal_acc_pct']:.2f}%, "
        f"Delta={r['delta_ann_minus_svc_pct']:+.2f} pp, "
        f"Winner={r['winner']}"
    )

lines.append("")
lines.append(f"ANN wins: {int((per_subject_table['winner'] == 'ANN').sum())}")
lines.append(f"SVC wins: {int((per_subject_table['winner'] == 'SVC').sum())}")
lines.append(f"Ties: {int((per_subject_table['winner'] == 'Tie').sum())}")
lines.append(f"Mean Delta (ANN-SVC): {per_subject_table['delta_ann_minus_svc_pct'].mean():+.2f} pp")
lines.append(f"Paired t-test p: {summary_df.loc[0, 'paired_t_p']:.6g}")
lines.append(f"Wilcoxon p: {summary_df.loc[0, 'wilcoxon_p']:.6g}")

report_path = OUT_DIR / "per_subject_report.txt"
report_path.write_text("\n".join(lines), encoding="utf-8")

print("Saved:", report_path)
print("")
print("\n".join(lines[:8]))

In [ ]:
print("Saved files in:", OUT_DIR)
for p in sorted(OUT_DIR.glob("*")):
    print("-", p.name)